In [1]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from useful_functions import *
import pandas as pd
from joblib import Parallel, delayed

In [2]:
spectra_data    = fits.open('/Users/hyp0515/data/0715_Spring_BGS_ALL_trimmed.fits')
color_data      = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_with_Flux.fits')
cigale_data     = fits.open('/Users/hyp0515/data/IronPhysProp_v1.2_extracted.fits')
fastspecfit     = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_fastspecfit.fits')

ids = read_ids('parent_samples_ids.txt')

SPECTRA = Spectrum(spectra_data, color_data, cigale_data, fastspecfit, load_targetID=None)


In [3]:
SPECTRA = SPECTRA.subtype_filter(subtype='QSO', exclude=True)
# SPECTRA.shrink_dataset(50)
SPECTRA.stack_data()
SPECTRA.mask_bad()

In [4]:
FIT     = FitSpectrum()
SPECTRA = FIT.shift_to_rest_frame(SPECTRA)
SPECTRA = FIT.label_emission_lines(SPECTRA, 3)
print(SPECTRA.n_spectra)


99831


In [5]:
SPECTRA = FIT.significant_emission_filter(SPECTRA)
print(SPECTRA.n_spectra)

61905


In [ ]:
with open('parent_samples_ids.txt', 'w+') as f:
    for tid in SPECTRA.df['TARGETID'].to_list():
        f.write(f'{tid}\n')

In [7]:
SPECTRA.df.head(5)

,TARGETID,RA,DEC,SPECTYPE,FLUX_G,FLUX_R,FLUX_Z,LOGM,LOGSFR,z_pipe,z,OII,Hbeta,OIII,Halpha,NII,SII
0,39627739380056197,177.003822,-1.903397,GALAXY,7.252433,17.472439,29.587103,10.250155,-0.348563,0.224113,0.224113,False,False,False,True,True,False
1,39627739380056420,177.013472,-1.972766,GALAXY,7.954081,19.348692,40.818554,9.722920,-0.349153,0.102005,0.102005,False,False,False,True,False,False
2,39627739380056564,177.019312,-1.965308,GALAXY,8.337815,20.474783,36.728188,10.452706,0.941032,0.323187,0.323187,True,False,False,True,True,False
3,39627739380056768,177.028143,-1.960056,GALAXY,25.459047,51.294968,96.631752,10.143390,0.193251,0.102814,0.102814,True,True,False,True,False,True
4,39627739380056864,177.032977,-1.909970,GALAXY,11.503003,30.335419,61.082424,10.575973,-9.632146,0.167324,0.167324,False,False,False,True,True,False


In [8]:
def process_target(target_id):
    """
    Processes a single target to find double-peaked features.
    """
    # try:
    p_value, delta_dv, dp_detection, line_fluxes_rank, params_2comp = FIT.find_dp(SPECTRA, id=target_id)

    dp_cols = ['OII3726_dp', 'OII3729_dp',
            'Hbeta_dp',
            'OIII4959_dp', 'OIII5007_dp',
            'NII6548_dp', 'Halpha_dp', 'NII6583_dp', 
            'SII6716_dp', 'SII6731_dp']

    dp_rank_cols = [f'{col[:-3]}_rank' for col in dp_cols]

    data = {
        'TARGETID': target_id,
        'p_value': p_value,
        'dv': params_2comp['dv'],
        'delta_dv': delta_dv,
        'sigma': params_2comp['sigma'],
    }
    data.update(dict(zip(dp_cols, dp_detection)))
    data.update(dict(zip(dp_rank_cols, line_fluxes_rank)))
    return data
    # except Exception as e:
    #     print(f"Error processing target {target_id}: {e}")
    #     return None

    

# Use joblib to parallelize the processing over all target IDs
# n_jobs=-1 uses all available CPU cores.
results = Parallel(n_jobs=10)(delayed(process_target)(target_id) for target_id in tqdm(SPECTRA.targetID))

# Convert the list of dictionaries to a DataFrame
dp_df = pd.DataFrame(results)

# display(dp_df)

100%|██████████| 61905/61905 [06:05<00:00, 169.60it/s]


In [9]:
display(dp_df.head(10))

,TARGETID,p_value,dv,delta_dv,sigma,OII3726_dp,OII3729_dp,Hbeta_dp,OIII4959_dp,OIII5007_dp,...,OII3726_rank,OII3729_rank,Hbeta_rank,OIII4959_rank,OIII5007_rank,NII6548_rank,Halpha_rank,NII6583_rank,SII6716_rank,SII6731_rank
0,39627739380056197,0.670855,"(15.713979292030979, -83.74977603898279)",99.463755,"(43.27801968315071, 28.63513437289793)",False,False,False,False,False,...,-1,-1,-1,-1,-1,2,0,1,-1,-1
1,39627739380056420,0.206380,"(44.028876160377855, -21.983613621870365)",66.012490,"(0.010000000008640314, 32.578531569552744)",False,False,False,False,False,...,-1,-1,-1,-1,-1,2,0,1,-1,-1
2,39627739380056564,0.000047,"(54.23557963405434, -77.21487029651152)",131.450450,"(50.13363863752554, 31.151569463609025)",False,False,False,False,False,...,3,1,-1,-1,-1,4,0,2,-1,-1
3,39627739380056768,0.045512,"(3.6917048319503856, -130.6857610163866)",134.377466,"(44.611683053915876, 0.01000000000000449)",False,False,False,False,False,...,3,1,4,-1,-1,7,0,2,5,6
4,39627739380056864,0.348877,"(4.887532420076487, -181.64711932413007)",186.534652,"(86.72374957200758, 53.83220908944886)",False,False,False,False,False,...,-1,-1,-1,-1,-1,2,0,1,-1,-1
5,39627739380057310,0.664107,"(7.579482800213949, -13.979119364843106)",21.558602,"(32.881476999428436, 0.010000000027719913)",False,False,False,False,False,...,2,1,-1,7,3,8,0,6,5,4
6,39627739380057506,0.000199,"(120.70568266484035, -57.385545803837516)",178.091228,"(39.7098954588015, 73.00516391253304)",False,False,False,False,False,...,-1,-1,-1,-1,-1,2,0,1,-1,-1
7,39627739380057763,0.000057,"(9.887676808472259, -49.219207385776954)",59.106884,"(63.36387778400713, 119.8573507327382)",False,False,False,False,False,...,-1,-1,3,-1,-1,2,0,1,-1,-1
8,39627739380057802,0.829053,"(67.90658958432667, -25.782735743000167)",93.689325,"(0.010063286739753704, 190.088922219885)",False,False,False,False,False,...,1,0,-1,-1,-1,-1,-1,-1,-1,-1
9,39627739380058028,0.035712,"(15.950162988043449, -6.314953462837355)",22.265116,"(0.010000000033095393, 37.82279666175752)",False,False,True,False,False,...,-1,-1,2,-1,-1,4,0,1,3,5


In [10]:
dp_df.to_csv('dp_parent_results.csv', index=False)

In [11]:
dp_df = pd.read_csv('dp_parent_results.csv', low_memory=False)
print(f'All: {len(dp_df)}')
dp_candidates = dp_df[(dp_df['p_value'] < 0.05) & (dp_df['delta_dv'] > 75)]
print(f'DP Candidates: {len(dp_candidates)} ({len(dp_candidates) / len(dp_df) * 100:.2f}%)')

All: 61905
DP Candidates: 26116 (42.19%)
